# AutoML Pipeline

Демонстрация нового AutoML слоя: `ModelSelector` запускает несколько моделей,
сравнивает по val-метрике и возвращает `AutoMLResult` с лучшей моделью.

**Модели в сравнении:** SeasonalNaive, CatBoost

**Pipeline:**
1. Загрузка данных
2. Создание сплитов
3. Конфигурация AutoML
4. Запуск `ModelSelector`
5. Сравнение моделей
6. Детальные метрики лучшей модели
7. (Опционально) Hyperopt через Optuna

In [ ]:
import sys
from pathlib import Path

import pandas as pd

sys.path.append("../..")

from src.automl.config import AutoMLConfig
from src.automl.selector import ModelSelector
from src.configs.settings import Settings
from src.model_selection import temporal_panel_split_by_size
from src.visualization.visualization import (
    plot_best_predictions,
    plot_overall_metrics_comparison,
    plot_panel_metrics_distributions,
    plot_worst_predictions,
)

settings = Settings()
data_dir = Path("../../data")

## 1. Загрузка данных

In [ ]:
mirrors_ts_df = pd.read_csv(data_dir / "processed" / "filtered_mirrors_ts.csv")
mirrors_ts_df["date"] = pd.to_datetime(mirrors_ts_df["date"])

print(f"Статей: {mirrors_ts_df['article'].nunique()}")
print(f"Период: {mirrors_ts_df['date'].min().date()} — {mirrors_ts_df['date'].max().date()}")
print(f"Строк: {len(mirrors_ts_df):,}")
mirrors_ts_df.head()

## 2. Создание сплитов

Последние 3 точки — test, предыдущие 3 — val, остальное — train.

In [ ]:
splits = temporal_panel_split_by_size(
    mirrors_ts_df,
    panel_column=settings.columns.id,
    time_column=settings.columns.date,
    test_size=3,
    val_size=3,
)

print(f"Train: {len(splits.train)} строк, {splits.train['date'].min().date()} — {splits.train['date'].max().date()}")
print(f"Val:   {len(splits.val)} строк, {splits.val['date'].min().date()} — {splits.val['date'].max().date()}")
print(f"Test:  {len(splits.test)} строк, {splits.test['date'].min().date()} — {splits.test['date'].max().date()}")

## 3. Конфигурация AutoML

In [ ]:
config = AutoMLConfig(
    models=["seasonal_naive", "catboost"],
    selection_metric="mape",
    use_hyperopt=False,
)

config

## 4. Запуск ModelSelector

In [ ]:
selector = ModelSelector(config)
result = selector.run(splits, settings)

print(f"\n✓ Лучшая модель: {result.best.name}")
print(f"  Метрика выбора: {result.selection_metric} на {result.selection_split}")

## 5. Сравнение всех моделей

In [ ]:
rows = []
for model_result in result.all_results:
    for split_eval in model_result.evaluation.splits:
        m = split_eval.overall_metrics
        rows.append({
            "model": model_result.name,
            "split": split_eval.split_name,
            "mape": round(m.mape, 4),
            "rmse": round(m.rmse, 2),
            "mae": round(m.mae, 2),
            "r2": round(m.r2, 4),
            "nrmse": round(m.nrmse, 4),
        })

comparison_df = pd.DataFrame(rows)
comparison_df.set_index(["model", "split"]).sort_index()

In [ ]:
val_comparison = (
    comparison_df[comparison_df["split"] == result.selection_split]
    .drop(columns=["split"])
    .sort_values(result.selection_metric)
    .reset_index(drop=True)
)

print(f"Сравнение на {result.selection_split} (сортировка по {result.selection_metric}):")
val_comparison.style.highlight_min(
    subset=["mape", "rmse", "mae", "nrmse"], color="lightgreen"
).highlight_max(subset=["r2"], color="lightgreen")

## 6. Детальные метрики лучшей модели

In [ ]:
best_eval = result.best.evaluation

print(f"Модель: {result.best.name}\n")
best_eval.get_overall_metrics_df().set_index("split").round(4)

In [ ]:
plot_overall_metrics_comparison(best_eval)

In [ ]:
plot_panel_metrics_distributions(
    results=best_eval,
    metrics_to_plot=["mape", "rmse", "r2"],
)

In [ ]:
plot_best_predictions(best_eval, n_best=5, metric="mape", split_name="test")

In [ ]:
plot_worst_predictions(best_eval, n_worst=5, metric="mape", split_name="test")

## 7. Распределение MAPE по панелям: обе модели

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, model_result in zip(axes, result.all_results):
    panel_df = model_result.evaluation.get_panel_metrics_df()
    test_mape = panel_df[panel_df["split"] == "test"]["mape"]
    ax.hist(test_mape.clip(upper=2.0), bins=30, edgecolor="white")
    ax.axvline(test_mape.median(), color="red", linestyle="--", label=f"Медиана: {test_mape.median():.3f}")
    ax.axvline(test_mape.mean(), color="green", linestyle="--", label=f"Среднее: {test_mape.mean():.3f}")
    ax.set_title(f"{model_result.name} — MAPE (test)")
    ax.set_xlabel("MAPE")
    ax.legend()

plt.tight_layout()
plt.show()

## 8. Hyperopt (опционально)

Подбор гиперпараметров CatBoost через Optuna. Занимает несколько минут.

In [ ]:
config_hyperopt = AutoMLConfig(
    models=["seasonal_naive", "catboost"],
    selection_metric="mape",
    use_hyperopt=True,
    n_trials=20,
)

selector_hyperopt = ModelSelector(config_hyperopt)
result_hyperopt = selector_hyperopt.run(splits, settings)

print(f"Лучшая модель: {result_hyperopt.best.name}")
print(f"Параметры: {result_hyperopt.best.params}")

In [ ]:
rows_hyperopt = []
for label, res in [("без hyperopt", result), ("с hyperopt", result_hyperopt)]:
    cb_result = next(r for r in res.all_results if r.name == "catboost")
    for split_eval in cb_result.evaluation.splits:
        rows_hyperopt.append({
            "config": label,
            "split": split_eval.split_name,
            "mape": round(split_eval.overall_metrics.mape, 4),
            "rmse": round(split_eval.overall_metrics.rmse, 2),
            "r2": round(split_eval.overall_metrics.r2, 4),
        })

pd.DataFrame(rows_hyperopt).set_index(["config", "split"]).sort_index()